In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import numpy as np
import matplotlib.pyplot as plt
from Train_fun import train_fun
from scipy.interpolate import Rbf
from RK import solve_rk4_adaptive_fixed_dt
from RK import build_f_sparse_mixed
from Bulid_Library import build_polynomial_library
from SSSR import SSSR
from SSSR import linear_reg

In [2]:
def rrmse(x,y):
    return (np.mean((x-y)**2))**0.5/(np.mean(y**2))**0.5

# Data generation

In [3]:
import scipy.sparse as sp
import scipy.sparse.linalg as spla

def Burgers2D_data(a=0.8, w=1.0, 
                     nu=1.0/10000.0,
                     L=3.0, dx=0.1, dy=0.1,
                     T=1.5, dt=1.5/1500.0,
                     picard_tol=1e-8, picard_maxit=10,
                     verbose=False):
    # grid
    x = np.arange(-L, L + 1e-12, dx)
    y = np.arange(-L, L + 1e-12, dy)
    Nx, Ny = len(x), len(y)                 # = 61 × 61
    assert abs(x[-1]-L) < 1e-12 and abs(y[-1]-L) < 1e-12
    # interior sizes
    mx, my = Nx - 2, Ny - 2                 # unknowns only on interior
    N = mx * my

    X, Y = np.meshgrid(x, y, indexing='xy')
    r2 = X**2 + Y**2
    ic = a * np.exp(-r2 / w)                # a * exp(-||x||^2 / w)
    # field arrays with boundaries present (we keep boundaries = 0)
    u = np.zeros((Ny, Nx))
    v = np.zeros((Ny, Nx))
    u[1:-1, 1:-1] = ic[1:-1, 1:-1]
    v[1:-1, 1:-1] = ic[1:-1, 1:-1]

    # ---- build sparse 1D operators on interior (Dirichlet 0 bc handled implicitly)
    ex = np.ones(mx)
    ey = np.ones(my)

    # backward difference in x: (f_i - f_{i-1})/dx   (first interior row uses boundary=0)
    Dx_b = sp.diags([ex, -ex[:-1]], [0, -1], shape=(mx, mx)) / dx
    # backward difference in y
    Dy_b = sp.diags([ey, -ey[:-1]], [0, -1], shape=(my, my)) / dy

    # central-diff 1D Laplacian piece: (f_{i+1}-2f_i+f_{i-1})/h^2
    Tx = sp.diags([ex[:-1], -2*ex, ex[:-1]], [-1, 0, 1], shape=(mx, mx)) / (dx*dx)
    Ty = sp.diags([ey[:-1], -2*ey, ey[:-1]], [-1, 0, 1], shape=(my, my)) / (dy*dy)

    Ix = sp.eye(mx); Iy = sp.eye(my)

    # 2D operators via Kronecker
    Dxb = sp.kron(Iy, Dx_b, format='csr')      # backward ∂/∂x
    Dyb = sp.kron(Dy_b, Ix, format='csr')      # backward ∂/∂y
    Lap = sp.kron(Iy, Tx, format='csr') + sp.kron(Ty, Ix, format='csr')  # Δ

    I = sp.eye(N, format='csr')

    # time stepping
    nsteps = int(np.round(T / dt))
    times = [0.0]
    
    U_hist = [u.copy().reshape(1,-1)]
    V_hist = [v.copy().reshape(1,-1)]
    if verbose:
        print(f"Nx×Ny = {Nx}×{Ny}, interior N = {N}, steps = {nsteps}, dt = {dt:.6f}")

    def fld_to_vec(F):
        """extract interior and flatten to (N,) in row-major consistent with our kron layout"""
        return F[1:-1, 1:-1].ravel(order='C')

    def vec_to_fld(g):
        """put interior vector back to (Ny,Nx) with boundaries zero"""
        G = np.zeros((Ny, Nx))
        G[1:-1, 1:-1] = g.reshape((my, mx), order='C')
        return G

    for n in range(1, nsteps + 1):
        u_old = u.copy()
        v_old = v.copy()

        # Picard iterations to treat convection implicitly (freeze advective velocity)
        u_k = u_old.copy()
        v_k = v_old.copy()

        rhs_u = fld_to_vec(u_old)
        rhs_v = fld_to_vec(v_old)

        for it in range(picard_maxit):
            ua = fld_to_vec(u_k)
            va = fld_to_vec(v_k)

            # Build linear operator for each component:
            # (I + dt*(ua*Dxb + va*Dyb - nu*Lap)) * U^{n} = rhs
            A = I + dt * (sp.diags(ua) @ Dxb + sp.diags(va) @ Dyb - nu * Lap)

            u_new_vec = spla.spsolve(A, rhs_u)
            v_new_vec = spla.spsolve(A, rhs_v)

            u_new = vec_to_fld(u_new_vec)
            v_new = vec_to_fld(v_new_vec)

            # convergence check
            du = np.linalg.norm(u_new - u_k) / (np.linalg.norm(u_k) + 1e-14)
            dv = np.linalg.norm(v_new - v_k) / (np.linalg.norm(v_k) + 1e-14)
            if max(du, dv) < picard_tol:
                u_k, v_k = u_new, v_new
                if verbose and (it > 0):
                    print(f"step {n}/{nsteps}: Picard it={it}, relchg={max(du, dv):.2e}")
                break

            u_k, v_k = u_new, v_new
            if it == picard_maxit - 1 and verbose:
                print(f"step {n}: Picard not converged, relchg={max(du, dv):.2e}")

        # accept
        u, v = u_k, v_k
        times.append(n * dt)
        U_hist.append(u.copy().reshape(1,-1))
        V_hist.append(v.copy().reshape(1,-1))
    
    return np.vstack(U_hist),np.vstack(V_hist)

In [4]:
m = 151
mu1_number = 11
mu2_number = 11
Para_number = mu1_number * mu2_number
Para = np.array(np.meshgrid(np.linspace(0.6, 0.9, mu1_number), np.linspace(0.8, 1.2, mu2_number))).T.reshape(-1,2)
U_all = []

for point in Para:
    mu1,mu2 = point[0],point[1]
    U_alpha,V_alpha = Burgers2D_data(mu1,mu2)
    U_alpha = U_alpha[::10, :]
    U_all.append(U_alpha)

U = np.vstack(U_all)  

KeyboardInterrupt: 

# POD

In [26]:
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=40, algorithm='randomized', n_iter=5, random_state=42)
U_mean = np.mean(U, axis=0)
U_centered = U - U_mean
U_snapshots = U_centered.T
svd.fit(U_snapshots.T)  
Phi = svd.components_.T 
Sigma = svd.singular_values_  

latent_dim = 9
Phi_r = Phi[:,:latent_dim]
Z = np.transpose(np.dot(Phi_r.T, U_snapshots))  

U_reconstructed = np.dot(Z, Phi_r.T) + U_mean
print('reconstruct error:', rrmse(U_reconstructed,U))

reconstruct error: 0.00459060777320594


# SSSR 

In [27]:
dt = 1.5/(m-1)
dZ = np.zeros_like(Z)

for i in range(Para_number):
    start = i * m
    end = (i + 1) * m
    a_seg = Z[start:end, :]  # shape: (m, r)
    
    da_seg = np.zeros_like(a_seg)
    da_seg[1:-1] = (a_seg[2:] - a_seg[:-2]) / (2 * dt)
    da_seg[0] = (a_seg[1] - a_seg[0]) / dt
    da_seg[-1] = (a_seg[-1] - a_seg[-2]) / dt
    
    dZ[start:end, :] = da_seg

In [28]:
include_functions = False
include_bias=False
degree = 1
Theta, feature_names = build_polynomial_library(Z, degree=degree, include_bias=include_bias, include_functions=include_functions)
sparsity_level = 4 
Z_pred = Z.copy()
coef_matrix = np.zeros((Para_number,(sparsity_level+1)*latent_dim))
Supports = np.zeros((latent_dim,sparsity_level))

for k in range(latent_dim):
    loss = 0
    support = SSSR(Theta, dZ[:,k].reshape(-1,1), sparsity_level=sparsity_level, Para_number=Para_number)
    print('Support set for latent variable {}:'.format(k), support)
    Supports[k,:] = support
    Library = []
    for j in range(Para_number):
        Target = dZ[m*j:m*(j+1),k].reshape(-1,1)
        state = Z[m*j:m*(j+1),k].reshape(-1,1)
        Feature = Theta[m*j:m*(j+1),support]
        reg, pred = linear_reg(Feature,Target)
        coef_matrix[j,(sparsity_level+1)*k] = reg.intercept_
        coef_matrix[j,(sparsity_level+1)*k+1:(sparsity_level+1)*(k+1)] = reg.coef_
        loss = loss + rrmse(pred,Target)
        state_next_pred = state + dt * pred
        state_pred = state.copy()
        state_pred[1:] = state_next_pred[:-1]
        Library.append(state_pred)
    print('Regression error of latent variable {}:'.format(k), loss/Para_number)
    state_pred = np.vstack(Library)
    Z_pred[:,k] = np.squeeze(state_pred)
    Supports = Supports.astype('int')

Support set for latent variable 0: [2, 1, 4, 6]
Regression error of latent variable 0: 0.0009119153805378042
Support set for latent variable 1: [0, 3, 4, 2]
Regression error of latent variable 1: 0.003082459921591509
Support set for latent variable 2: [0, 4, 3, 5]
Regression error of latent variable 2: 0.00168313774421373
Support set for latent variable 3: [0, 4, 3, 6]
Regression error of latent variable 3: 0.006636634548746606
Support set for latent variable 4: [2, 5, 1, 3]
Regression error of latent variable 4: 0.004829654522843891
Support set for latent variable 5: [4, 3, 6, 0]
Regression error of latent variable 5: 0.02321855166736053
Support set for latent variable 6: [5, 0, 6, 1]
Regression error of latent variable 6: 0.07389735050393849
Support set for latent variable 7: [8, 5, 3, 1]
Regression error of latent variable 7: 0.09975238681703566
Support set for latent variable 8: [6, 7, 0, 1]
Regression error of latent variable 8: 0.06765088374650972


In [29]:
Library = []
for j in range(Para_number):
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=coef_matrix[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.00001)
    Library.append(Y)
Z_pred_multi_step = np.vstack(Library)

In [30]:
print('The prediction error of Z by multiple step:', rrmse(Z_pred_multi_step,Z))
U_pred_multi_step = np.dot(Z_pred_multi_step, Phi_r.T) + U_mean  # shape = (m, n)
print('The prediction error of X by multiple step:', rrmse(U_pred_multi_step,U)) 

The prediction error of Z by multiple step: 0.013799841052625229
The prediction error of X by multiple step: 0.005717079656094221


# Parameter-to-coefficient mapping

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.interpolate import Rbf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [32]:
Para_data = np.hstack((Para,coef_matrix))

## RBF

In [33]:
X = Para_data[:, 0:2]        
Y = Para_data[:, 2:] 
N, r = Y.shape

# -----------------------------
# 2. 为每个低维系数单独构建 RBF 插值器
# -----------------------------
rbf_models = []
for j in range(r):
    rbf = Rbf(X[:,0], X[:,1], Y[:, j], function='multiquadric')
    rbf_models.append(rbf)

## NN

In [34]:
X = Para_data[:, 0:2]        
Y = Para_data[:, 2:]        

X_train_t = torch.from_numpy(X).float().to(device)
Y_train_t = torch.from_numpy(Y).float().to(device)
output_dim = Y.shape[1]

In [35]:
class FourNet_sin(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, 128)
        self.linear2 = nn.Linear(128, 64)
        self.linear3 = nn.Linear(64,64)
        self.linear4 = nn.Linear(64,output_dim)

    def forward(self, x):
        x = torch.sin(self.linear1(x))
        x = torch.sin(self.linear2(x))
        x = torch.sin(self.linear3(x))
        x = self.linear4(x)
        return x

model = FourNet_sin(input_dim=2, output_dim=output_dim).to(device)

In [36]:
loss, time, loss_hist = train_fun(model,X_train_t,Y_train_t,N_red_lr=4,epochs=5000,lr=0.001,threshold=0.0001,printfun=False)

运行时间： 24.22674059867859
loss: 0.0028751306235790253


# Online predict 

## Set test parameter point

In [37]:
Para_test = np.array([[0.754,0.987]])
Para_test_number = len(Para_test)

U_test_all = []

for point in Para_test:
    mu1,mu2 = point[0],point[1]
    U_alpha,V_alpha = Burgers2D_data(mu1, mu2)
    U_alpha = U_alpha[::10,:]
    U_test_all.append(U_alpha)

U_test = np.vstack(U_test_all)  

## RBF

In [38]:
# get latent initial condiction
U_centered_test = U_test - U_mean
U_snapshots_test = U_centered_test.T 
Z_test = np.dot(Phi_r.T, U_snapshots_test).T

In [39]:
# predcit the coefficients
Y_test_pred_RBF = []
for j, rbf in enumerate(rbf_models):
    Y_test_pred_RBF.append(rbf(Para_test[:,0],Para_test[:,1]).reshape(-1,1))
Y_test_pred_RBF = np.hstack(Y_test_pred_RBF)

# sloving the latent dynamical system 
Library = []
for j in range(Para_test_number):   
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=Y_test_pred_RBF[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z_test[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.001)
    Library.append(Y)
Z_pred_multi_step_RBF = np.vstack(Library)

# reconstruction 
U_pred_multi_step_RBF = np.dot(Z_pred_multi_step_RBF, Phi_r.T) + U_mean  # shape = (m, n)

In [40]:
rrmse(U_pred_multi_step_RBF,U_test)

0.004706564061096854

## NN

In [41]:
# predcit the coefficients
model.eval()
with torch.no_grad():
    Y_test_pred_NN = model(torch.from_numpy(Para_test).float().to(device)).cpu().numpy()

# sloving the latent dynamical system 
Library = []
for j in range(Para_test_number):   
    f, info = build_f_sparse_mixed(feature_names , Supports=Supports, 
                              coef_matrix=Y_test_pred_NN[j].reshape(latent_dim,-1), latent_dim=latent_dim)
    z0 =  Z_test[m*j,:]
    T, Y = solve_rk4_adaptive_fixed_dt(f, 0, dt*(m-1), z0, dt, rtol=1e-6, atol=1e-9, h0=1e-2, h_min=0.001)
    Library.append(Y)
Z_pred_multi_step_NN = np.vstack(Library)

# reconstruction 
U_pred_multi_step_NN= np.dot(Z_pred_multi_step_NN, Phi_r.T) + U_mean  # shape = (m, n)

In [42]:
rrmse(U_pred_multi_step_NN,U_test)

0.005645949727950461